# Introduction

This is the first Playground competition this year where we have time-series problem! Despite seemingly similar, this is extremely different compared to all problems we have faced so far! Using a neural network might actually be a legit strategy due to the existence of RNN architectures!

In this competition, we have to forecast sales of each product from each store in each country in the entirety of 2022. We are using historical data from January 1st, 2017 until December 31st, 2021 to train our models.

# Loading Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from category_encoders import OneHotEncoder, MEstimateEncoder, GLMMEncoder, OrdinalEncoder, CatBoostEncoder
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold, TimeSeriesSplit
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor, StackingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.linear_model import HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, roc_auc_score, roc_curve
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.calibration import CalibratedClassifierCV
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

sns.set_theme(style = 'white', palette = 'colorblind')
pal = sns.color_palette('colorblind')

pd.set_option('display.max_rows', 100)

In [ ]:
train = pd.read_csv(r'../input/playground-series-s3e19/train.csv')
test_1 = pd.read_csv(r'../input/playground-series-s3e19/test.csv')

train.drop('id', axis = 1, inplace = True)
test = test_1.drop('id', axis = 1)

# Descriptive Statistics

Let's know our data descriptively first before diving in! We start by taking a peek at the top 10 row of each dataset, and then see the statistics of each features.

In [ ]:
train.head(10)

It seems that all of our stores are from Kaggle! While the products being sold are mainly books about LLM.

In [ ]:
desc = pd.DataFrame(index = list(train))
desc['count'] = train.count()
desc['nunique'] = train.nunique()
desc['%unique'] = desc['nunique'] / len(train) * 100
desc['null'] = train.isnull().sum()
desc['type'] = train.dtypes
desc = pd.concat([desc, train.describe().T.drop('count', axis = 1)], axis = 1)
desc

Surprisingly, despite the fact that there are 136950 samples, we only have 1028 unique values of number of products sold.

In [ ]:
test.head(10)

Test dataset seems to be similar to train dataset here.

In [ ]:
desc = pd.DataFrame(index = list(test))
desc['count'] = test.count()
desc['nunique'] = test.nunique()
desc['%unique'] = desc['nunique'] / len(test) * 100
desc['null'] = test.isnull().sum()
desc['type'] = test.dtypes
desc = pd.concat([desc, test.describe().T.drop('count', axis = 1)], axis = 1)
desc

Country, store, and product in test have same exact amount of unique value with train.

In [ ]:
categorical_features = ['country', 'store', 'product']
numerical_features = test.drop(categorical_features, axis = 1).columns

# Feature Recasting

Let's start by casting the date as `datetime`. We want to be able to visualize the sales overtime before doing anything!

In [ ]:
train.date = pd.to_datetime(train.date, format = '%Y-%m-%d')
test.date = pd.to_datetime(test.date, format = '%Y-%m-%d')

# Sales Distribution

Let's see the distribution of the sales first.

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.kdeplot(data = train, x = 'num_sold', fill = True)
    
plt.title('Sales Distribution', fontsize = 24, fontweight = 'bold')
plt.show()

It looks like our target is skewed to the left.

# Sales Growth Over Time

Now let's start visualizing our sales over time. We can try spotting any trends visible in the visualization! We begin by visualizing general sales over time.

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.lineplot(data = train, x = 'date', y = 'num_sold', errorbar = None)
    
plt.title('Sales Over Time', fontsize = 24, fontweight = 'bold')
plt.show()

There is a consistent jump in each end of year or beginning of new year. This can be explained simply by Christmas and New Year holidays. Both are important holidays celebrated around the world. There seems to be also consistent jump in the middle of the year, but we can't try guessing just from this visualization alone. We will go back to it later.

Another thing that we can notice is the massive slump in 2020. You may remember a lot of pandemic era where a lot of people have to stay on home. This causes a massive economic downturn around the world, and our problem isn't exception to this.

Now let's see what happens if we try to visualize sales by countries.

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.lineplot(data = train, x = 'date', y = 'num_sold', hue = 'country', errorbar = None)
    
plt.title('Sales Over Time per Country', fontsize = 24, fontweight = 'bold')
plt.show()

We can see which countries have the biggest sales and which countries have the lowest sales. In order from biggest to lowest, those countries are Canada, Japan, Spain, Estonia, and Argentina.

One very interesting pattern that we can see however, lies in Japan. It looks like our consistent jump in the middle of the year is due to an event in Japan. If I have to guess, the jump is due to Golden Week. From April 29th to May 5th, Japan celebrates holiday after holiday, as if merging them into one where everyone celebrates holidays for an entire week for everyone.

Let's see the trend by store now.

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.lineplot(data = train, x = 'date', y = 'num_sold', hue = 'store', errorbar = None)
    
plt.title('Sales Over Time per Store', fontsize = 24, fontweight = 'bold')
plt.show()

So far, there doesn't seem to be much difference from previous trend other than overall difference between stores. We can see that Kagglazon has the biggest sales, followed by Kaggle Store and finally Kaggle Learn as the lowest selling store.

Finally, let's try visualizing sales trends by product.

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.lineplot(data = train, x = 'date', y = 'num_sold', hue = 'product', errorbar = None)
    
plt.title('Sales Over Time per Product', fontsize = 24, fontweight = 'bold')
plt.show()

Finally we have a new trend to see! We can see one product consistently having low demand over the years and also lower than all other products. Other products, on the other hand, seems to have seasonal trend. `Using LLMs to Write Better` for example, have higher sales in Spring and lower sales in Fall, while the opposite seems to be true for `Using LLMs to Improve Your Coding`.

# Distribution of Categorical Feature

We want to know the proportion of each categorical feature and also the comparison between train and test dataset. Usually, if both dataset have similar distribution in tabular data, we can know that we can trust our CV. However, this is time-series, it might still be different from our previous CV process.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    train['country'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 5)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = train, y = 'country', ax = ax[1], palette = 'viridis', order = train['country'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Country in Train Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    test['country'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 5)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = test, y = 'country', ax = ax[1], palette = 'viridis', order = test['country'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Country in Test Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    train['store'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 3)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = train, y = 'store', ax = ax[1], palette = 'viridis', order = train['store'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Store in Train Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie( 
    test['store'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 3)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = test, y = 'store', ax = ax[1], palette = 'viridis', order = test['store'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Store in Test Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    train['product'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 5)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = train, y = 'product', ax = ax[1], palette = 'viridis', order = train['product'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Product in Train Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    test['product'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 5)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = test, y = 'product', ax = ax[1], palette = 'viridis', order = test['product'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Product in Test Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

All categorical features are perfectly symmetrical. It might seem surprising, but it's not, since this is time-series data.

# Metrics

This competition uses Symmetrical Mean Absolute Percentage Error (SMAPE) to evaluate the result. The formula is as follows:

$${SMAPE} = \frac{100}{n} \sum_{t=1}^n \frac{\left|F_t-A_t\right|}{(|A_t|+|F_t|)/2}$$

We can translate it into Python function as follows:

In [ ]:
def smape(A, F):
    return 100/len(A) * np.sum(2 * np.abs(F - A) / (np.abs(A) + np.abs(F)))

# Preparation

One interesting thing about SMAPE is that you can approximate the result by using log-transformation on the target, and then use MAE to get the result. Creating a custom loss function takes more effort without guarantee that it will be successful.

In [ ]:
X = train.copy()
y = X.pop('num_sold')
y = np.log1p(y)

seed = 42
k = TimeSeriesSplit(n_splits = 4, test_size = 27390)

np.random.seed(seed)

# Date Processor

To make any future adjustment easier, I'll create a class to extract features from date. This class will then be put inside the model pipeline to automate the process.

In [ ]:
class DateProcessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, x, y = None): 
        return self
    def transform(self, x, y = None):
        x_copy = x.copy()
        x_copy['day'] = x_copy.date.dt.day
        x_copy['month'] = x_copy.date.dt.month
        x_copy['year'] = x_copy.date.dt.year
        x_copy['dow'] = x_copy.date.dt.dayofweek
        x_copy = x_copy.drop('date', axis = 1)
        return x_copy        

# Post-processor

Due to the massive difference in the distribution of test dataset, I will use post-processor to level th sales between countries.

In [ ]:
def multipliers(predictors, prediction, canada = 1, japan = 1, spain = 1, estonia = 1, argentina = 1):
    prediction[predictors.country == 'Canada'] *= canada
    prediction[predictors.country == 'Japan'] *= japan
    prediction[predictors.country == 'Spain'] *= spain
    prediction[predictors.country == 'Estonia'] *= estonia
    prediction[predictors.country == 'Argentina'] *= argentina
    return prediction

# Cross-Validation Function

This cross-validation function is built in a way so we can detect overfitting easily.

In [ ]:
def cross_val_score(model, cv = k, label = ''):
    
    X = train.copy()
    y = X.pop('num_sold')
    y = np.log1p(y)
    
    #initiate prediction arrays and score lists
    val_predictions = np.zeros((len(train)))
    #train_predictions = np.zeros((len(train)))
    train_scores, val_scores = [], []
    
    #training model, predicting prognosis probability, and evaluating log loss
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        #define train set
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        #define validation set
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        #train model
        model.fit(X_train, y_train)
        
        #make predictions
        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)
                  
        val_predictions[val_idx] += val_preds
        
        #reverse log-transformation
        y_train = np.expm1(y_train)
        y_val = np.expm1(y_val)
        
        train_preds = np.expm1(train_preds)
        val_preds = np.expm1(val_preds)
        
        #evaluate model for a fold
        train_score = smape(y_train, train_preds)
        val_score = smape(y_val, val_preds)
        
        #append model score for a fold to list
        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')
    
    return val_scores, val_predictions

# Model

I'll start by building a comparing different gradient boosting models, excluding XGBoost since it performs terribly when using MAE as the loss function. As has been said before, we can use MAE to approximate SMAPE. Therefore, I will use `mae` or `absolute_error` as loss function for any gradient boosting models. As for encoding, I will use GLMMEncoder.

In [ ]:
score_list, oof_list = pd.DataFrame(), pd.DataFrame()

models = [
    ('lgb', LGBMRegressor(random_state = seed, objective = 'mae')),
    ('cb', CatBoostRegressor(random_state = seed, verbose = 0, objective = 'MAE')),
    ('gb', GradientBoostingRegressor(random_state = seed, loss = 'absolute_error')),
    ('hgb', HistGradientBoostingRegressor(random_state = seed, loss = 'absolute_error'))
]

Encoder = GLMMEncoder(cols = ['country', 'store', 'product'], random_state = seed)

In [ ]:
for (label, model) in models:
     score_list[label], oof_list[label] = cross_val_score(
         make_pipeline(DateProcessor(), Encoder, model),
         label = label,
     )

In [ ]:
plt.figure(figsize = (8, 4), dpi = 300)
sns.barplot(data = score_list.reindex((score_list).mean().sort_values().index, axis = 1), palette = 'viridis', orient = 'h')
plt.title('Score Comparison', weight = 'bold', size = 20)
plt.show()

# Retraining

I will retrain the model on the entire dataset this time to get the optimal result.

In [ ]:
model = make_pipeline(DateProcessor(), Encoder, HistGradientBoostingRegressor(random_state = seed, loss = 'absolute_error'))

model.fit(X, y)
prediction = model.predict(test)

# Submission

In [ ]:
test_1.drop(list(test_1.drop('id', axis = 1)), axis = 1, inplace = True)

test_1['num_sold'] = multipliers(test, np.expm1(prediction) * 1.5, .58, .76, 1, 1.08, 2.82)
test_1.to_csv('submission.csv', index = False)

In [ ]:
plt.figure(figsize = (20, 10), dpi = 300)

sns.lineplot(x = test.date, y = test_1.num_sold, hue = test.country, errorbar = None)
    
plt.title('Predicted Sales Over Time', fontsize = 24, fontweight = 'bold')
plt.show()

Thank you for reading!